In [0]:

dbutils.widgets.text("batch_id", "BATCH_02")
dbutils.widgets.dropdown(
    "phase",
    "after_merge",
    ["before_merge", "after_merge"]
)

batch_id = dbutils.widgets.get("batch_id")
phase = dbutils.widgets.get("phase")

print(f"Batch: {batch_id}, phase: {phase}")

# Table names
silver_table = "hospital_appointments.silver.silver_data"
audit_table = "hospital_appointments.bronze.batch_reconciliation"

batch_sql = batch_id.replace("'", "''")

def count_rows(table):
    return spark.sql(
        f"SELECT COUNT(*) AS n FROM {table}"
    ).first()["n"]

# AFTER MERGE
if phase == "after_merge":

    saved = spark.sql(f"""
        SELECT *
        FROM {audit_table}
        WHERE batch_id = '{batch_sql}'
    """).first()

    if saved is None:
        raise Exception(
            f"No pre-merge audit record found for {batch_id}."
        )

    if saved["status"] != "PENDING":
        raise Exception(
            f"Expected PENDING audit status, got {saved['status']}."
        )

    actual_count = count_rows(silver_table)
    expected_count = saved["expected_silver_count"]

    status = "PASS" if actual_count == expected_count else "FAIL"

    spark.sql(f"""
        UPDATE {audit_table}
        SET
            actual_silver_count = {actual_count},
            status = '{status}',
            checked_at = current_timestamp()
        WHERE batch_id = '{batch_sql}'
    """)

    print(f"Expected Silver rows: {expected_count}")
    print(f"Actual Silver rows:   {actual_count}")
    print(f"Validation status:    {status}")

    if status != "PASS":
        raise Exception(
            f"Silver validation failed: expected {expected_count}, got {actual_count}"
        )
